In [1]:
RUN_MODE = "observed-dev"
CONTRACT_VERSION = "2.1.2"
CRAWL_RELEASE_ID = "CRAWL_20260806_03"
DATA_VERSION = "observed-dev-20260806.1"
AS_OF_DATE = "2026-08-06"
RANDOM_SEED = 42
DATA_PROVENANCE = "OBSERVED_DEVELOPMENT_ONLY"
EMPIRICAL_ANALYSIS_ALLOWED = False
PROMOTION_ALLOWED = False
DUTY_INPUT_PATH = ""
GOLD_INPUT_PATH = ""
CONTROL_SCHEMA_DIR = ""

# P4 Agent 4 · Core AI·IT Code Set

**Stage:** `A4-01-CODESET` · **Mode:** `observed-dev` · **Contract:** `2.1.2`

Deterministically rebuild and review the 120-code set: 69 included and 51 excluded.

> Development-only orchestration. Empirical analysis and production promotion are disabled.

In [2]:
from pathlib import Path
import os
import sys
import pandas as pd

NCS_ROOT = Path.cwd().resolve()
if NCS_ROOT.name != 'ncs_mapping':
    raise RuntimeError('run this notebook with cwd=ncs_mapping')
sys.path.insert(0, str(NCS_ROOT / 'src'))
assert RUN_MODE == 'observed-dev'
assert DATA_PROVENANCE == 'OBSERVED_DEVELOPMENT_ONLY'
assert EMPIRICAL_ANALYSIS_ALLOWED is False and PROMOTION_ALLOWED is False
resolved_duty_input = DUTY_INPUT_PATH or os.environ.get('P4_A2_DUTY_HANDOFF', '')
resolved_gold_input = GOLD_INPUT_PATH or os.environ.get('P4_NCS_GOLD_INPUT', '')
resolved_schema_dir = CONTROL_SCHEMA_DIR or os.environ.get('P4_CONTROL_SCHEMA_DIR', '')

In [3]:
from p4_ncs.codeset.core_ai_it import build_core_ai_it_codeset

ncs_units = pd.read_parquet(NCS_ROOT / 'data/processed/ncsUnit.parquet')
rebuilt_codeset = build_core_ai_it_codeset(ncs_units)
input_audit = {
    'total': len(rebuilt_codeset),
    'included': int(rebuilt_codeset['included'].sum()),
    'excluded': int((~rebuilt_codeset['included']).sum()),
    'codeSetStatus': 'REVIEW_REQUIRED',
}
assert input_audit == {'total': 120, 'included': 69, 'excluded': 51, 'codeSetStatus': 'REVIEW_REQUIRED'}
input_audit

{'total': 120,
 'included': 69,
 'excluded': 51,
 'codeSetStatus': 'REVIEW_REQUIRED'}

In [4]:
from p4_ncs.workflow.observed import run_stage

stage_manifest = run_stage('A4-01-CODESET', root=NCS_ROOT, duty_input_path=resolved_duty_input or None, gold_input_path=resolved_gold_input or None, schema_dir=resolved_schema_dir or None)
stage_manifest

{'manifestVersion': 'stage-manifest-v1',
 'runId': 'NCS_MAPPING_OBSERVED_20260806_01',
 'runMode': 'observed-dev',
 'stageId': 'A4-01-CODESET',
 'status': 'SUCCEEDED',
 'agentId': 'P4-A4-NCS',
 'branch': 'agent/p4-ncs-mapping-v2',
 'gitHead': 'cce6067e578cb8dc99aacaeb465439cb1ef0faa1',
 'contractVersion': '2.1.2',
 'schemaVersion': 'core-ai-it-v0.1',
 'dataVersion': 'observed-dev-20260806.1',
 'crawlReleaseId': 'CRAWL_20260806_03',
 'dataProvenance': 'OBSERVED_DEVELOPMENT_ONLY',
 'startedAt': '2026-08-06T08:38:42.287670Z',
 'completedAt': '2026-08-06T08:38:42.290632Z',
 'empiricalAnalysisAllowed': False,
 'promotionAllowed': False,
 'inputManifestSha256': 'b5b02ea232295931f245c69c881046a813f04cfbc517a13bc40d9e82994277c1',
 'parameterSha256': 'ee2a95b4cf722fbc15def7f7c955eba2248f681ec2a54ffef33e1869c3836e36',
 'rowCounts': {'coreCodeSet': 120, 'included': 69, 'excluded': 51},
 'gateResults': [{'gateId': 'NCS_CODESET_REVIEW_READY',
   'status': 'PASS',
   'evidencePath': 'ncs_mapping/dat

In [5]:
stage_root = NCS_ROOT / 'data/runs' / RUN_MODE / 'NCS_MAPPING_OBSERVED_20260806_01' / stage_manifest['stageId']
expected_artifacts = {'stage_manifest.json', 'stage_metrics.json', 'stage_quality.csv', 'CHECKSUMS.sha256'}
actual_artifacts = {path.name for path in stage_root.iterdir() if path.is_file()}
assert actual_artifacts == expected_artifacts
termination_summary = {'stageId': stage_manifest['stageId'], 'status': stage_manifest['status'], 'rowCounts': stage_manifest['rowCounts'], 'artifacts': sorted(actual_artifacts)}
termination_summary

{'stageId': 'A4-01-CODESET',
 'status': 'SUCCEEDED',
 'rowCounts': {'coreCodeSet': 120, 'included': 69, 'excluded': 51},
 'artifacts': ['CHECKSUMS.sha256',
  'stage_manifest.json',
  'stage_metrics.json',
  'stage_quality.csv']}